# Arrays

Notes and runnable examples on the array family of data structures in Python — from the raw contiguous-memory idea, up through `list` and its internals, the more static-leaning options, and a guide to when each one fits.

**Contents**

1. **Static arrays** — the fixed-size, contiguous baseline
2. **Dynamic arrays** — size vs. capacity, amortized growth
3. **Python's `list`** — internals, growth strategy, and the aliasing gotcha
4. **Beyond `list`** — `tuple`, `array`, `numpy`, `deque`
5. **When to use what** — a quick decision guide
6. **Complexity cheat-sheet** — Big-O across all five

## 1. Static arrays

fixed size, contiguous block of memory allocated at once. In a language like C, `int arr[10]` reserves exactly 10 integers' worth of memory in a row. Because the elements sit contiguously and each has a known size, accessing index i is just pointer arithmetic: `base_address + i * element_size`. That gives O(1) random access. The catch is that you can't grow it, if you need an 11th element, you need to create a whole other array and copy the stuff over.

## 2. Dynamic arrays

A dynamic array hides this limitation. It works by tracking the size and capacity. So if you append when there's spare capacity left (unused), it is cheap, and expensive when you append and you are full, as it then creates a bigger block of memory, copies everything over, and frees the old.

## 3. Python's `list`

### Internals

python's built in `list` is a dynamic array. Under the hood (CPython) it is a struct called `PyListObject` that holds a pointer to a contiguous C array, plus the current size and allocated capacity. One important wrinkle: that C array doesn't store your objects inline, it stores the pointers to them. Every element is a `PyObject*` (8 bytes on a 64-bit build) pointing to the actual integer, string, or whatever, living elsewhere on the heap. That's why a python list can hold mixed types and why indexing is still O(1), you jump to a slot and follow the pointer.

### Growth strategy

CPython's growth strategy isn't a clean doubling like you may have seen before. When it needs new room it computes the new capacity (in `list_resize` in `listobject.c`) as `new_size + (new_size >> 3) + 6`, then rounds that result *down* to a multiple of 4 (`& ~3`). The `>> 3` is the key term — adding `new_size / 8` works out to a growth factor of about 1.125x. The documented growth pattern is `0, 4, 8, 16, 24, 32, 40, 52, 64, 76, …`; you can watch the start of it below.

In [1]:
import sys

# sizeof numbers below reflect a 64-bit CPython 3.x build:
# empty list = 56 bytes, each extra slot = 8 bytes (one PyObject*)
lst = []
prev = -1
for i in range(20):
    cap = sys.getsizeof(lst)
    if cap != prev:
        print(f"len={len(lst):2d} cap={cap:4d}")
        prev = cap
    lst.append(i)

len= 0 cap=  56
len= 1 cap=  88
len= 5 cap= 120
len= 9 cap= 184
len=17 cap= 248


The byte count jumps in steps rather than rising smoothly with each append — each jump is a reallocation. The smaller growth factor trades a few more reallocations for less wasted memory, which suits Python where lists tend to be small.

### Gotcha: aliasing & copying

A `list` holds **references**, not values — and two operations that *look* like they copy actually share the inner objects. This is the source of some of Python's nastiest bugs.

**`[inner] * n` repeats the reference, not the object.** `[[0] * 3] * 3` builds *one* inner list and points at it three times, so mutating one "row" mutates them all:

In [2]:
grid = [[0] * 3] * 3          # three references to the SAME inner list
grid[0][0] = 1                # ...so this writes through all three rows
print("after grid[0][0] = 1:", grid)
print("rows are the same object:", grid[0] is grid[1] is grid[2])

grid = [[0] * 3 for _ in range(3)]   # the fix: a fresh inner list each iteration
grid[0][0] = 1
print("with a comprehension:", grid)
print("rows are distinct   :", grid[0] is not grid[1])

after grid[0][0] = 1: [[1, 0, 0], [1, 0, 0], [1, 0, 0]]
rows are the same object: True
with a comprehension: [[1, 0, 0], [0, 0, 0], [0, 0, 0]]
rows are distinct   : True


**Slicing and `copy.copy` are *shallow*.** `lst[:]` or `copy.copy(lst)` duplicate the *outer* list, but the inner objects are still shared — only `copy.deepcopy` recurses all the way down:

In [3]:
import copy

a = [[1, 2], [3, 4]]
shallow = a[:]                # copies the outer list; the inner lists are shared
shallow[0][0] = 99
print("after shallow-copy edit:", a)     # a leaked the change -> [[99, 2], [3, 4]]

a = [[1, 2], [3, 4]]
deep = copy.deepcopy(a)       # recursively copies everything
deep[0][0] = 99
print("after deep-copy edit   :", a)     # a is untouched -> [[1, 2], [3, 4]]

after shallow-copy edit: [[99, 2], [3, 4]]
after deep-copy edit   : [[1, 2], [3, 4]]


## 4. Beyond `list`: more static-leaning options

Python doesn't give you a true fixed-size, compile-time array like C's `int arr[10]`, but several types sit closer to that end of the spectrum. They differ by what they trade away — a packed typed buffer, immutability, or fast operations at both ends. We'll take them one at a time, each with a quick demo.

### `array` — packed C primitives, and the memory payoff

The `array` module (`array.array('i', [...])`) stores actual C primitives contiguously rather than pointers, so it's type-restricted and far more memory-compact for large numeric sequences. It can still grow, though, so it's "static" only in element type.

A `list`, by contrast, stores `PyObject*` pointers, and the int objects they point to live separately on the heap — so you pay for both. `array` and `numpy` pack the raw C values contiguously with no per-element object overhead. The difference is easy to measure:

> **Caveat on the number below:** the "list + int objs" total sums `getsizeof` over every element, which counts the small ints `0`–`256` as if each were distinct. CPython actually caches those as shared singletons, so the figure is a slight over-count — but the per-element pointer-plus-object overhead it illustrates is real.

In [4]:
import sys
from array import array
import numpy as np

n = 1000
py_list = list(range(n))
py_arr  = array('i', range(n))      # 'i' = C signed int, 4 bytes each
np_arr  = np.arange(n, dtype=np.int32)

# a list also pays for the int objects its pointers point to
list_total = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list)

print(f"list (container only):  {sys.getsizeof(py_list):>7} bytes")
print(f"list (+ the int objs):  {list_total:>7} bytes")
print(f"array('i') buffer:      {sys.getsizeof(py_arr):>7} bytes")
print(f"numpy int32 buffer:     {np_arr.nbytes:>7} bytes")

list (container only):     8056 bytes
list (+ the int objs):    36056 bytes
array('i') buffer:         4200 bytes
numpy int32 buffer:        4000 bytes


### `tuple` — immutable, but still an array of references

A *tuple* is immutable: it has a fixed size for life, you can't append or grow it. It's still an array of pointers internally, so it's not about memory efficiency, it's about immutability. And fixing the *tuple's* size doesn't freeze the objects it points to — only the slots are locked, not what lives at the other end of each pointer.

In [5]:
t = (1, 2, 3)
try:
    t[0] = 99
except TypeError as e:
    print("immutable:", e)

# the tuple is fixed, but a mutable element it points to can still change
box = ([1, 2],)
box[0].append(3)
print(box)        # ([1, 2, 3],) — the tuple didn't change, the list it references did

immutable: 'tuple' object does not support item assignment
([1, 2, 3],)


### `numpy.ndarray` — typed, contiguous buffer

`numpy.ndarray` is the closest to a real static array for numeric work: a contiguous, typed, fixed-size buffer. Resizing it generally means allocating a new array. This is why numpy is so fast — no per-element pointer chase and no Python-level loop; operations run as tight C loops over that packed buffer.

In [6]:
import numpy as np
import timeit

N = 1_000_000
py = list(range(N))
nd = np.arange(N)

t_py = timeit.timeit(lambda: [x * 2 for x in py], number=1)
t_np = timeit.timeit(lambda: nd * 2,              number=1)
print(f"python list comp: {t_py:.4f}s")
print(f"numpy vectorized: {t_np:.4f}s  (~{t_py/t_np:.0f}x faster)")

python list comp: 0.0239s
numpy vectorized: 0.0022s  (~11x faster)


### `collections.deque` — fast at both ends

Because list appends are amortized O(1) but inserts at the front (`list.insert(0, x)`) force every existing element to shift, front insertion is O(n). If you need fast operations at both ends, `collections.deque` is the right tool instead — `appendleft`/`popleft` are O(1). The trade-off is that indexing into the middle becomes O(n), since a deque is a chain of fixed-size blocks rather than one contiguous buffer.

In [7]:
from collections import deque
import timeit

N = 20_000

def list_front():
    xs = []
    for i in range(N):
        xs.insert(0, i)        # shifts every element -> O(n) per op

def deque_front():
    dq = deque()
    for i in range(N):
        dq.appendleft(i)       # O(1) per op

t_list  = timeit.timeit(list_front,  number=1)
t_deque = timeit.timeit(deque_front, number=1)
print(f"list.insert(0):   {t_list:.4f}s")
print(f"deque.appendleft: {t_deque:.4f}s  (~{t_list/t_deque:.0f}x faster)")

list.insert(0):   0.0218s
deque.appendleft: 0.0003s  (~69x faster)


## 5. When to use what

| You need… | Reach for | Why |
|---|---|---|
| Flexible, general-purpose sequence of anything | `list` | dynamic array of references; mixed types; amortized O(1) append |
| A fixed, hashable record that never changes | `tuple` | immutable; hashable (if its elements are), so usable as a dict key / set member |
| Compact storage of many numbers, still growable | `array` | packed C primitives, one dtype, far less memory than `list` |
| Fast numeric math over big buffers | `numpy.ndarray` | contiguous typed buffer, vectorized C-loop ops |
| Fast push/pop at *both* ends (queues, sliding windows) | `collections.deque` | O(1) at both ends (but O(n) random access) |

**Mental model:** `list` for the flexible, general-purpose dynamic array of references; `array`/`numpy` when you want the packed, typed, near-static layout and the performance that comes with it; `deque` when your access pattern lives at the ends, not the middle.

## 6. Complexity cheat-sheet

| Operation | `list` | `tuple` | `array` | `deque` | `numpy` |
|---|:---:|:---:|:---:|:---:|:---:|
| Index access | O(1) | O(1) | O(1) | O(n) | O(1) |
| Append (end) | O(1)\* | — | O(1)\* | O(1) | O(n)† |
| Pop (end) | O(1) | — | O(1) | O(1) | — |
| Insert / pop front | O(n) | — | O(n) | O(1) | O(n) |
| Insert / delete middle | O(n) | — | O(n) | O(n) | O(n) |
| Search (unsorted) | O(n) | O(n) | O(n) | O(n) | O(n) |
| Memory per element | high (ptr + obj) | high (ptr + obj) | low (C type) | high (ptr + obj) | low (C type) |

<sub>\* amortized — an occasional O(n) reallocation when capacity is exceeded &middot; † `np.append` allocates a whole new array &middot; `deque` is a doubly-linked list of fixed-size blocks, so random indexing is O(n) and it stores boxed-object pointers like `list` &middot; `tuple` is immutable, so size-changing ops simply don't apply.</sub>